In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git  = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

In [ ]:
estimate = 'ACS5'
sample_type = 'PUMS_p'
years_to_import = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
geography = 'PUMA'
variables = ','.join(['SERIALNO', 'PUMA', 'RT', 'HISP', 'RAC1P', 'PWGTP'])
# variables = ','.join(['SERIALNO', 'PUMA', 'HISP', 'RAC1P', 'GRPIP', 'OCPIP', 'WGTP', 'PWGTP'])

state = '06'
puma = ','.join(['01700', '06102', '06103', '06101', '06707', '06714'])#, '06709', '06703', '06704', '06705', '06706', '06701', '06712', '06713', '06717'])

In [ ]:
list_df_acs = []

for year in tqdm(years_to_import):
    try:
        df_pums = query_acs(api_Key     = api_key
                            , estimate  = estimate
                            , sample    = sample_type
                            , geography = geography
                            , variables = variables
                            , year      = year
                            , state     = state
                            , puma      = puma)
                
        list_df_acs.append(df_pums)
                
    except:
        pass
            
df_acs_raw = pd.concat(list_df_acs)

In [ ]:
# view raw data
pd.set_option('display.max_columns', None)
print(df_acs_raw.shape)
print(df_acs_raw.Year.unique())
df_acs_raw.head(3)

In [ ]:
df_acs = df_acs_raw.copy()
df_acs['WGTP' ] = df_acs['WGTP' ].apply(pd.to_numeric)
df_acs['PWGTP'] = df_acs['PWGTP'].apply(pd.to_numeric)

In [ ]:
df_acs[df_acs['SERIALNO'] == '2018HU0000480']

In [ ]:
df_acs[df_acs['SERIALNO'] == '2018HU0001095']

In [ ]:
df_acs[df_acs['SERIALNO'].str.contains('HU0422727')]

In [ ]:
df_acs[(df_acs['WGTP'] > 0) & (df_acs['PWGTP'] > 0)]